<a href="https://colab.research.google.com/github/ysasson-portfolio/text-analytics-spring-2026/blob/main/assignment_5/notebooks/Yarden_Sasson_A5_OptionB_Job_Fit_Starter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5 — Option B: Job Fit Analyzer
## BSAN 6200: Text Mining & Social Media Analytics — Spring 2026

**Student Name:** Yarden Sasson
**Date:** May 13, 2026  
**Option:** B — Job Fit Analyzer  
**API Path:** Paid

---

### Table of Contents
1. [Setup and Imports](#1-setup)
2. [Load Job Descriptions and Resume](#2-loading)
3. [Text Chunking](#3-chunking)
4. [Embedding and Vector Store](#4-embedding)
5. [Analysis Prompts and Chain](#5-analysis)
6. [Zero-shot vs. Few-shot Comparison](#6-comparison)
7. [Evaluation](#7-evaluation)

> **Reminder:** The Streamlit app is a separate file (`streamlit_app.py`). This notebook builds and tests the analysis pipeline.  
> See the Option B Implementation Guide for detailed step requirements.

---
<a id="1-setup"></a>
## 1. Setup and Imports

Install required packages and load your API key from a `.env` file.  
**Do NOT hardcode API keys in this notebook.**

Suggested packages: `langchain`, `langchain-openai` or `langchain-community`, `chromadb` or `faiss-cpu`, `pypdf`, `python-dotenv`, `pandas`, `sentence-transformers` (free path)

In [28]:
# ── Install packages (uncomment as needed) ──
!pip install langchain langchain-openai chromadb pypdf python-dotenv sentence-transformers
!pip install -q langchain langchain-community
# ── Load API keys from .env ──
import os
import pandas as pd
from dotenv import load_dotenv
load_dotenv()
from langchain_core.documents import Document
import requests

# ── Your imports below ──

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-exporter-otlp-proto-common==1.38.0, but you have opentelemetry-exporter-otlp-proto-common 1.41.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-proto==1.38.0, but you have opentelemetry-proto 1.41.1 which is incompatible.
opentelemetry-exporter-otlp-proto-http 1.38.0 requires opentelemetry-sdk~=1.38.0,

In [31]:
#Clone the Repository (only needs to happen once)
!git clone https://github.com/ysasson-portfolio/text-analytics-spring-2026.git

Cloning into 'text-analytics-spring-2026'...
remote: Enumerating objects: 517, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 517 (delta 15), reused 50 (delta 9), pack-reused 458 (from 1)
Receiving objects: 100% (517/517), 40.41 MiB | 15.06 MiB/s, done.
Resolving deltas: 100% (254/254), done.


In [58]:
#If the github gets updated and I want the latest changes without re-cloning
%cd /content/text-analytics-spring-2026
!git pull

/content/text-analytics-spring-2026
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 2), reused 6 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 3.68 KiB | 1.23 MiB/s, done.
From https://github.com/ysasson-portfolio/text-analytics-spring-2026
   0e94629..9965a4b  main       -> origin/main
Updating 0e94629..9965a4b
Fast-forward
 ...nalyst, Strategy and Analytics-Sofi Stadium.txt | 84 ++++++++++++++++++++++
 1 file changed, 84 insertions(+)


---
<a id="2-loading"></a>
## 2. Load Job Descriptions and Resume

**Required:**
- 10+ JD files in `data/job_descriptions/` (each as a separate .txt or .pdf)
- Your resume in `data/resume/`
- A metadata file `data/jd_metadata.csv` with columns: filename, company, title, source_url, date_collected

Print: number of JDs loaded, number of resume docs, and preview content from each.

In [59]:
# ── Load JD metadata ──
metadata_df = pd.read_csv("https://raw.githubusercontent.com/ysasson-portfolio/text-analytics-spring-2026/refs/heads/main/assignment_5/data/jd_metadata.csv")

print(metadata_df)

                                           Job Title  \
0                      Business Intelligence Analyst   
1  Business Intelligence Analyst, Sports - Brand ...   
2                      Business Intelligence Analyst   
3                                   Business Analyst   
4                                   Business Analyst   
5                                   Business Analyst   
6                                 Business Analyst I   
7                Sr. Analyst, Strategy and Analytics   
8                                 Strategy Associate   
9                                  Manager, Strategy   

                                    Company  \
0                             Guitar Center   
1                   Creative Artists Agency   
2  Los Angeles Tourism and Convention Board   
3             Red Bull Distribution Company   
4                                   Hadrian   
5                        Polestar Analytics   
6                  Skyworks Solutions, Inc.   
7      

In [60]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

folder_path = "/content/text-analytics-spring-2026/assignment_5/data/job_descriptions"

loader = DirectoryLoader(
    folder_path,
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)

documents = loader.load()

print(len(documents))

10


In [61]:
documents[0]

Document(metadata={'source': '/content/text-analytics-spring-2026/assignment_5/data/job_descriptions/Business Analyst-Polestar Analytics.txt'}, page_content="About the job\nJob Title: Business Analyst - Retail\n\nLocation: Los Angeles\n\nJob Type: Full-time\n\nExperience: 2+ years\n\nIndustry: Analytics Services\n\n\n\nRoles and Responsibilities:\n\nWork closely with clients and internal stakeholders to gather, analyse, and document business requirements.\nTranslate business needs into functional specifications, user stories, and process flows with a strong focus on data-driven decision making.\nSupport consulting engagements by conducting market research, competitor benchmarking, and industry analysis.\nCollaborate with Product, Data Engineering, and Analytics teams to define solution approaches aligned with client objectives.\nAssist in creating business cases, value propositions, and solution decks for client presentations.\nParticipate in workshops, stakeholder discussions, and req

In [62]:
job_rows = []

for doc in documents:

    # Get filename from path
    file_name = os.path.basename(doc.metadata["source"])

    # Remove .txt extension
    file_name = file_name.replace(".txt", "")

    job_rows.append({
        "File Name": file_name,
        "job_description": doc.page_content
    })

job_df = pd.DataFrame(job_rows)

job_df.head()

,File Name,job_description
0,Business Analyst-Polestar Analytics,About the job\nJob Title: Business Analyst - R...
1,Strategy Manager- Paramount,"The Manager, Strategy will support the develop..."
2,Business Analyst-RedBull,"In this role, the successful candidate will be..."
3,"Sr. Analyst, Strategy and Analytics-Sofi Stadium","About Hollywood Park\n\nHollywood Park, a near..."
4,Strategy Associate-Cedars Sinai,Align yourself with an organization that has a...


In [63]:
merged_df = job_df.merge(
    metadata_df,
    on="File Name",
    how="left"
)

merged_df

,File Name,job_description,Job Title,Company,Source URL,Date Collected,Role Type
0,Business Analyst-Polestar Analytics,About the job\nJob Title: Business Analyst - R...,Business Analyst,Polestar Analytics,https://www.linkedin.com/jobs/view/4399610949/,4/28/2026,Business Analyst
1,Strategy Manager- Paramount,"The Manager, Strategy will support the develop...","Manager, Strategy",Paramount,https://www.linkedin.com/jobs/view/4398583482/,4/28/2026,Strategy Analyst
2,Business Analyst-RedBull,"In this role, the successful candidate will be...",Business Analyst,Red Bull Distribution Company,https://www.linkedin.com/jobs/view/4405852876/,4/28/2026,Business Analyst
3,"Sr. Analyst, Strategy and Analytics-Sofi Stadium","About Hollywood Park\n\nHollywood Park, a near...","Sr. Analyst, Strategy and Analytics",SoFi Stadium and Hollywood Park,https://www.linkedin.com/jobs/view/4388281012/,4/28/2026,Strategy Analyst
4,Strategy Associate-Cedars Sinai,Align yourself with an organization that has a...,Strategy Associate,Cedars Sinai,https://www.linkedin.com/jobs/view/4400809577/,4/28/2026,Strategy Analyst
5,Business Intelligence Analyst-CAA,Who We Are \nCreative Artists Agency (CAA) is ...,"Business Intelligence Analyst, Sports - Brand ...",Creative Artists Agency,https://www.linkedin.com/jobs/view/4402204259/,4/28/2026,Business Intelligence Analyst
6,Business Analyst-Skyworks,If you are looking for a challenging and excit...,Business Analyst I,"Skyworks Solutions, Inc.",https://www.linkedin.com/jobs/view/4403514344/,4/28/2026,Business Analyst
7,Business Analyst-Hadrian,Hadrian is building autonomous factories that ...,Business Analyst,Hadrian,https://www.linkedin.com/jobs/view/4402613474/,4/28/2026,Business Analyst
8,Business Inteligence Analyst-Guitar Center,"About the Role: \nAt Guitar Center, the Data ...",Business Intelligence Analyst,Guitar Center,https://www.linkedin.com/jobs/collections/reco...,4/28/2026,Business Intelligence Analyst
9,Business Intelligence Analyst- LA Tourism and ...,WHO WE ARE\n\nThe mission of the Los Angeles T...,Business Intelligence Analyst,Los Angeles Tourism and Convention Board,https://www.linkedin.com/jobs/view/4366060859/,4/28/2026,Business Intelligence Analyst


In [64]:
merged_df["job_description"]

,job_description
0,About the job\nJob Title: Business Analyst - R...
1,"The Manager, Strategy will support the develop..."
2,"In this role, the successful candidate will be..."
3,"About Hollywood Park\n\nHollywood Park, a near..."
4,Align yourself with an organization that has a...
5,Who We Are \nCreative Artists Agency (CAA) is ...
6,If you are looking for a challenging and excit...
7,Hadrian is building autonomous factories that ...
8,"About the Role: \nAt Guitar Center, the Data ..."
9,WHO WE ARE\n\nThe mission of the Los Angeles T...


In [ ]:
# ── Preview sample content ──


---
<a id="3-chunking"></a>
## 3. Text Chunking

Split your documents into chunks.  
**Required:** Try at least 2 chunking strategies, compare them quantitatively, and justify your final choice.

**Hint:** JDs often have natural sections (Requirements, Responsibilities, Qualifications). Consider whether your splitter respects these boundaries.

In [ ]:
# ── Strategy 1 ──


In [ ]:
# ── Strategy 2 ──


In [ ]:
# ── Compare strategies ──


### Chunking Decision

**Which strategy did you choose?**  
**Why?**  
**Final settings (chunk_size, overlap):**

---
<a id="4-embedding"></a>
## 4. Embedding and Vector Store

Embed your chunks and store them in a vector database (ChromaDB or FAISS).

**Paid path:** OpenAI `text-embedding-3-small`  
**Free path:** `sentence-transformers/all-MiniLM-L6-v2`

After creating the store, run a test similarity search to verify it works.

In [ ]:
# ── Create embeddings and vector store ──


In [ ]:
# ── Verify: run a test similarity search ──


---
<a id="5-analysis"></a>
## 5. Analysis Prompts and Chain

Build 3 analysis types, each with its own prompt (one iteration for 3 types) or 3 iterations for 1 type with its own prompt:

1. **Skill Gap Report:** Compare resume skills vs. JD requirements. Output matching skills, missing skills, and recommended actions.
2. **Keyword Alignment:** Extract key terms from a JD, check which appear in the resume, report a match rate.
3. **Fit Summary:** 3-4 sentence narrative assessment citing evidence from both documents.

You also need to wire up the LLM and a way to pass a specific JD + resume into each prompt.

**Required:** Document at least 3 prompt iterations total (across any analysis type) with rationale.

**Reminder:** Prompt design must be your own work (Tier 2 — AI prohibited for this step).

In [ ]:
# ── Initialize LLM ──


In [ ]:
# ── Analysis 1: Skill Gap Report ──
#Give me a analysis of the skill gap between the resume inputted and the job description (Baseline)
#Compare and Contrast the resume to the job posting and let me know where the skill gap needs to improve (Iteration 1)
#Explain what is missing from my resume and why it is important for this position (Iteration 2)
#If I have the skills for this position, do I have enough experience for the position as well? (Iteration 3) (Potentially add more and be aware of the changes)

In [ ]:
# ── Analysis 2: Keyword Alignment ──


In [ ]:
# ── Analysis 3: Fit Summary ──


### Prompt Iteration Log

Document at least 3 total iterations across any of the analysis types.

**Iteration 1:** [Which analysis? What changed? Why? What improved?]

**Iteration 2:** [Which analysis? What changed? Why? What improved?]

**Iteration 3:** [Which analysis? What changed? Why? What improved?]

---
<a id="6-comparison"></a>
## 6. Zero-shot vs. Few-shot Comparison

Pick one of your 3 analysis types. Create a few-shot version by adding 1-2 example input/output pairs to the prompt. Run both versions on the same JD and compare outputs.

**Reminder:** You must write the few-shot examples yourself (Tier 2).

In [ ]:
# ── Few-shot version of your chosen analysis ──
# Based on the following job description, identify whether the job is match from the resume.

#Job Description: (insert job description here)
#Match: Yes because ....

#Job Description: (insert job description here)
#Match: No because ....

In [ ]:
# ── Run both on the same JD, display side by side ──


### Zero-shot vs. Few-shot Analysis

**Which analysis type did you compare?**

**Which performed better?**

**Why? (use specific examples from the outputs above)**

---
<a id="7-evaluation"></a>
## 7. Evaluation

Run all 3 analysis types on your **top 3 target JDs** (9 total analyses).

For each, score:
- **Retrieval relevance:** Did it pull the right JD sections? (Yes/Partial/No)
- **Skill identification accuracy:** Are identified skills/gaps correct? (count correct vs. incorrect)
- **Actionability:** Are recommendations specific and useful? (1-5)
- **Faithfulness:** Does output stick to document content? (Faithful/Partial/Hallucinated)

**Reminder:** Evaluation must be your own work (Tier 2 — AI prohibited).

In [ ]:
# ── Run 9 analyses (3 JDs x 3 analysis types) ──


In [ ]:
# ── Summarize evaluation results ──


### Evaluation Analysis

**Which analysis type worked best?**

**Which JDs produced the best/worst results? Why?**

**Where did the system hallucinate or produce inaccurate results?**

**What would you improve?**

---

## Next Steps

1. Build your Streamlit app (`streamlit_app.py`) using the pipeline from this notebook
2. Write your Technical Manager Memo (`memo.md`)
3. Complete your AI Usage Log (`ai_log.md`)
4. Verify GitHub repository structure and commit count

---
*BSAN 6200 | Spring 2026 | Assignment 5 — Option B*